# Evaluation


## Setup

In [8]:
# Magic Codes 
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# IMPORTS

from features import engineer_features
from cleaning import clean_data

import wgnd as wg  
import joblib

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, HTML

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, ConfusionMatrixDisplay

In [10]:
# SETTINGS

# Global Notebook Style
display(HTML("<style>table { margin-left: 0 !important; }</style>"))

# Global Pandas Options
pd.options.display.float_format = '{:.4f}'.format
pd.set_option('display.memory_usage', 'deep')

In [11]:
# CONSTANTS

DATA_PATH_RAW = '../data/01_raw/'

DATA_PATH_INPUT  = '../data/02_interim/'
DATA_PATH_OUTPUT = '../data/03_processed/'
MODEL_PATH       = '../data/04_models/LogReg_Base_Simple8.joblib'

tracker = wg.ModelTracker()    

## Best Model Loading

In [12]:
# LOADING MODEL

wg.print_header('Loading Final Model')

best_model = joblib.load(MODEL_PATH)
print(f"Favorite Modell erfolgreich aus \n{MODEL_PATH} \ngeladen.")

wg.print_footer()




~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
LOADING FINAL MODEL
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Favorite Modell erfolgreich aus 
../data/04_models/LogReg_Base_Simple8.joblib 
geladen.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~




# Predict TEST Data 

In [13]:
# PROCESS TEST DATA

X_test_interim = pd.read_parquet(DATA_PATH_INPUT+'features_test.parquet')
y_test = pd.read_parquet(DATA_PATH_INPUT+'target_test.parquet')

# 3. PROCESSING (Exakt die gleiche Kette wie bei Train!)
# Schritt A: Cleaning
X_test_cleaned = clean_data(X_test_interim)

# Schritt B: Feature Engineering (Hier entstehen feat_price_ratio etc.)
X_test_final = engineer_features(X_test_cleaned)

# 4. SAVE TO PROCESSED
wg.save_processed_data(X_test_final, y_test, folder=DATA_PATH_OUTPUT, name='test')

print("✅ Test-Set synchronisiert und bereit für Evaluation!")





# LOADING DATA

wg.print_header('Loading Test Data')

DATA_PATH_PROCESSED = '../data/03_processed/'

X_test = pd.read_parquet(f'{DATA_PATH_PROCESSED}X_test_final.parquet')
y_test = pd.read_parquet(f'{DATA_PATH_PROCESSED}y_test.parquet').iloc[:, 0]

print(f"Daten erfolgreich geladen.")
print(f"Features: {X_test.shape[0]} Zeilen | {X_test.shape[1]} Spalten")
print(f"Target:   '{y_test.name}' (BadBuy-Rate: {y_test.mean():.2%})")

wg.print_footer()





# PREDICTION

wg.print_header("Final Prediction with Test-Data")

y_pred_test = best_model.predict(X_test)

print(classification_report(y_test, y_pred_test))

wg.print_footer()

 DATA CLEANING & SUCCESS REPORT 
KAPITEL 1: COMPLETENESS
  - Erkannt:  26602 kategoriale Lücken
  - Aktion:   26602 gefüllt mit 'Unknown'

KAPITEL 2: INTEGRITY
  - Erkannt:    175 Logik-Fehler (< 100$, Alter, Odo)
  - Aktion:     175 geheilt via Imputation
  - Aktion:       0 unrettbar gelöscht
-------------------------------------------------------
ENDERGEBNIS:
  - Datensätze (Rows):  13124
  - Retention Rate:    100.00%
  - Churn Rate:          0.00%

 FEATURE ENGINEERING REPORT 
Status: Enrichment Complete
Added Features (5):
  [+] feat_price_ratio
  [+] feat_price_diff
  [+] feat_market_trend
  [+] feat_miles_per_year
  [+] feat_warranty_ratio
Total Columns now: 37

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
PROCESSED TEST DATA EXPORT
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
🚀 X-Features: 13124 rows | 37 cols
🎯 y-Labels:   Gespeichert (13124 Zeilen)
📂 Location:   ../data/03_processed/
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
✅ Test-Set synchronisiert und bereit für Evaluation!



~~~

# Predict AIM Data

In [22]:
# ==========================================
# PREDICT AIM DATA (THE FINALE)
# ==========================================

wg.print_header('Predict AIM Data')

# 1. LOADING
wg.print_title('Loading Data')
X_aim_raw = pd.read_csv('../data/01_raw/features_aim.csv', sep=';') # ACHTUNG: Check sep=';' oder ','
print(f"Daten erfolgreich geladen: {X_aim_raw.shape[0]} Zeilen")

# 2. PROCESSING
wg.print_title('Cleaning Data')
X_aim_cleaned = clean_data(X_aim_raw)

wg.print_title('Engineering Features')
X_aim_final = engineer_features(X_aim_cleaned)

# 3. PREDICT TARGET
wg.print_title("Final Prediction with AIM-Data")

# Wir nutzen direkt X_aim_final aus dem Speicher
y_pred_aim = best_model.predict(X_aim_final)

# 4. EXPORT RESULTS
predictions_df = pd.DataFrame({'IsBadBuy': y_pred_aim})
predictions_df.to_csv('../data/05_results/predictions_aim.csv', index=False)





wg.print_title("Summary of AIM-Predictions")

summary_df = predictions_df['IsBadBuy'].value_counts().reset_index()
summary_df.columns = ['Status', 'Count']
summary_df['Label'] = summary_df['Status'].map({0: '✅ Good Buys', 1: '⚠️ Bad Buys'})
summary_df['Percentage'] = (summary_df['Count'] / len(predictions_df) * 100).round(2)
display_df = summary_df[['Label', 'Count', 'Percentage']].copy()
display_df['Percentage'] = display_df['Percentage'].astype(str) + " %"

print(f"FINALE VORHERSAGE: {len(predictions_df)} Fahrzeuge analysiert")
wg.print_seperator()
print(display_df.to_string(index=False, justify='left'))
wg.print_seperator()

bad_count = summary_df.loc[summary_df['Status'] == 1, 'Count'].values[0]
bad_perc = summary_df.loc[summary_df['Status'] == 1, 'Percentage'].values[0]

print(f"Ergebnis: Dein Modell rät bei {bad_perc}% der Autos zur Vorsicht.")
print(f"Datei exportiert nach: ../data/05_results/predictions_aim.csv")

wg.print_footer()






~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
PREDICT AIM DATA
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Loading Data
----------------------------------------
Daten erfolgreich geladen: 7292 Zeilen

Cleaning Data
----------------------------------------
 DATA CLEANING & SUCCESS REPORT 
KAPITEL 1: COMPLETENESS
  - Erkannt:  14474 kategoriale Lücken
  - Aktion:   14474 gefüllt mit 'Unknown'

KAPITEL 2: INTEGRITY
  - Erkannt:     91 Logik-Fehler (< 100$, Alter, Odo)
  - Aktion:      91 geheilt via Imputation
  - Aktion:       0 unrettbar gelöscht
-------------------------------------------------------
ENDERGEBNIS:
  - Datensätze (Rows):   7292
  - Retention Rate:    100.00%
  - Churn Rate:          0.00%


Engineering Features
----------------------------------------
 FEATURE ENGINEERING REPORT 
Status: Enrichment Complete
Added Features (5):
  [+] feat_price_ratio
  [+] feat_price_diff
  [+] feat_market_trend
  [+] feat_miles_per_year
  [+] feat_warranty_ratio
Total Columns now: 37


Final